# Ethiopia Food Prices — 12 EXPLAINABILITY

Explain which variables drive the model decisions and how the final price and spike signals are formed.


In [6]:
import pandas as pd
from pathlib import Path
import joblib

root = Path('../')
feature_df = pd.read_csv(root / 'data' / 'eth_food_prices_features.csv', parse_dates=['date'])
feature_df = feature_df.dropna().copy()

forecast_feature_cols = [
    'year', 'month_of_year', 'lag_1', 'lag_3', 'lag_6',
    'rolling_mean_3', 'rolling_mean_6', 'rolling_mean_12',
    'rolling_std_3', 'rolling_std_6', 'rolling_std_12',
    'pct_change_1m', 'price_to_mean_ratio'
]

spike_feature_cols = [
    'year', 'month_of_year', 'lag_1', 'lag_3', 'lag_6',
    'rolling_mean_3', 'rolling_mean_6', 'rolling_mean_12',
    'rolling_std_3', 'rolling_std_6', 'rolling_std_12',
    'pct_change_1m', 'price_to_mean_ratio', 'price_outlier_flag'
]

forecast_model = joblib.load(root / 'models' / 'maize_forecast_model.joblib')
print(type(forecast_model).__name__)
print('Forecast feature importance ranking:')

if hasattr(forecast_model, 'feature_importances_'):
    importance = pd.Series(forecast_model.feature_importances_, index=forecast_feature_cols).sort_values(ascending=False)
    print(importance.head(10))
else:
    coef = pd.Series(forecast_model.coef_, index=forecast_feature_cols)
    print(coef.abs().sort_values(ascending=False).head(10))


LinearRegression
Forecast feature importance ranking:
price_to_mean_ratio    2.347593e+01
rolling_mean_6         5.712264e-15
rolling_mean_12        3.752943e-15
rolling_std_6          3.193875e-15
lag_3                  3.153013e-15
month_of_year          2.997602e-15
rolling_std_3          2.605690e-15
rolling_mean_3         2.301622e-15
lag_1                  1.578356e-15
lag_6                  1.137997e-15
dtype: float64


In [ ]:
# Explainability on spike classifier
classifier = joblib.load(root / 'models' / 'spike_classifier_model.joblib')
feature_df_class = feature_df.copy()
X = feature_df_class[spike_feature_cols]
y = feature_df_class['is_spike_flag'].astype(int)

if hasattr(classifier, 'feature_importances_'):
    imp = pd.Series(classifier.feature_importances_, index=spike_feature_cols).sort_values(ascending=False)
    print('\nSpike model feature importances:')
    print(imp.head(10))
else:
    coef = pd.Series(classifier.coef_.ravel(), index=spike_feature_cols)
    print('\nSpike model coefficients:')
    print(coef.abs().sort_values(ascending=False).head(10))



Spike model feature importances:
pct_change_1m          0.103645
price_to_mean_ratio    0.091479
year                   0.087710
lag_1                  0.081253
rolling_std_12         0.080351
month_of_year          0.073616
rolling_mean_12        0.067083
rolling_std_6          0.066528
rolling_mean_3         0.063733
rolling_mean_6         0.062473
dtype: float32


The most important variables are usually the recent price history and rolling market trend indicators. In practice, a spike is often explained by a sudden deviation from a commodity's normal recent path rather than a single isolated outlier.
